# Rag Pipeline -Data ingestion to Vector Data base Pipeline

In [2]:
import os
import numpy as np
import chromadb #store embeddings and perform similarity search
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

import uuid
from typing import List, Any

from sentence_transformers import SentenceTransformer #Convert text → numerical vectors (embeddings)
import chromadb #Store and search embeddings efficiently, retrieves similar vectors quickly
from chromadb.config import Settings #Controls things like: where data is stored and performance setting
import uuid # Generates unique IDs Each chunk must have a unique ID in the database:
from typing import List, Dict, Any, Tuple #


In [3]:
pdf_directory = "/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs"
pdf_dir = Path(pdf_directory) #1. Create a Path object
pdf_files = list(pdf_dir.glob("*.pdf")) #that searches for files matching a pattern inside a directory. “Find all files in this folder that match *.pdf”
pdf_files

[PosixPath('/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/0796_paper.pdf'),
 PosixPath('/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/Few-Shot Learnng From Gigapixel Images via Hierarchical Vision-Language Alignment and Modeling.pdf'),
 PosixPath('/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/Historical Report Guided Bi-modal Concurrent Learning for Pathology Report Generation.pdf'),
 PosixPath('/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/PathFLIP Fine-grained Language-Image Pretraining commented.pdf'),
 PosixPath('/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/WsiCaption Multiple Instance Generation of Pathology Reports for Gigapixel Whole-Slide Images.pdf'),
 PosixPath('/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/MAMMOTH.pdf'),
 PosixPath('/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/Pathology Report Generation and.pdf')]

In [5]:
load = PyPDFLoader(str(pdf_files[0])) #PyPDFLoader is a class that can read PDF files and extract their content. It takes the path to a PDF file as an argument and provides methods to access the text and metadata of the document.
doc = load.load()
doc[0].page_content[:500]

'HistGen: Histopathology Report Generation via\nLocal-Global Feature Encoding and Cross-modal\nContext Interaction\nZhengrui Guo1, Jiabo Ma1, Yingxue Xu1, Yihui Wang1, Liansheng Wang3, and\nHao Chen1,2(\x00)\n1Department of Computer Science and Engineering, The Hong Kong University of\nScience and Technology\njhc@cse.ust.hk\n2Department of Chemical and Biological Engineering, The Hong Kong University of\nScience and Technology\n3School of Information Science and Engineering, Xiamen University\nAbstract. Histop'

# Creating documents out of pdfs


In [6]:
#read all pdfs inside the directory and load them into a list of documents

def processing_pdfs(pdf_directory):
    "process all pdfs in the directory and return a list of documents"
    pdf_dir = Path(pdf_directory)  #1. Create a Path object
    pdf_files = list(pdf_dir.glob("*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files in the directory.")
    all_documents = []
    
    for pdf_file in pdf_files:
        print(f"processing {pdf_file}...")
        
        try:
            # Try loading with PyPDFLoader first
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name 
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"Successfully loaded {pdf_file} with PyPDFLoader.")
        except Exception as e:
            print(f"PyPDFLoader failed for {pdf_file} with error: {e}")
            print(f"Trying PyMuPDFLoader for {pdf_file}...")
            
            try:
                # If PyPDFLoader fails, try PyMuPDFLoader
                loader = PyMuPDFLoader(str(pdf_file))
                documents = loader.load()
                
                for doc in documents:
                    doc.metadata['source_file'] = pdf_file.name 
                    doc.metadata['file_type'] = 'pdf'
                
                all_documents.extend(documents)
                print(f"Successfully loaded {pdf_file} with PyMuPDFLoader.")
            except Exception as e2:
                print(f"PyMuPDFLoader also failed for {pdf_file} with error: {e2}")
            print(f"The total number of documents loaded so far: {len(all_documents)}")
    return all_documents


all_pdf_documents = processing_pdfs(pdf_directory)

Found 7 PDF files in the directory.
processing /data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/0796_paper.pdf...
Successfully loaded /data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/0796_paper.pdf with PyPDFLoader.
processing /data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/Few-Shot Learnng From Gigapixel Images via Hierarchical Vision-Language Alignment and Modeling.pdf...
Successfully loaded /data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/Few-Shot Learnng From Gigapixel Images via Hierarchical Vision-Language Alignment and Modeling.pdf with PyPDFLoader.
processing /data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/Historical Report Guided Bi-modal Concurrent Learning for Pathology Report Generation.pdf...
Successfully loaded /data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/Historical Report Guided Bi-modal Concurrent Learning for Pathology Report Generation.pdf with PyPDFLoader.
processing /data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/PathFLIP Fine-grained Language-Image Pretraining comm

In [5]:
all_pdf_documents[0].page_content[:500]

'HistGen: Histopathology Report Generation via\nLocal-Global Feature Encoding and Cross-modal\nContext Interaction\nZhengrui Guo1, Jiabo Ma1, Yingxue Xu1, Yihui Wang1, Liansheng Wang3, and\nHao Chen1,2(\x00)\n1Department of Computer Science and Engineering, The Hong Kong University of\nScience and Technology\njhc@cse.ust.hk\n2Department of Chemical and Biological Engineering, The Hong Kong University of\nScience and Technology\n3School of Information Science and Engineering, Xiamen University\nAbstract. Histop'

In [7]:
len(all_pdf_documents)

131

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len, separators=["\n\n", "\n", " ", ""]) # recursively chunk documents into smaller pieces based on the specified chunk size and overlap.
split_documents = text_splitter.split_documents(all_pdf_documents) # Split the documents into chunks using the text splitter.
split_documents[0].page_content[:500]

'HistGen: Histopathology Report Generation via\nLocal-Global Feature Encoding and Cross-modal\nContext Interaction\nZhengrui Guo1, Jiabo Ma1, Yingxue Xu1, Yihui Wang1, Liansheng Wang3, and\nHao Chen1,2(\x00)\n1Department of Computer Science and Engineering, The Hong Kong University of\nScience and Technology\njhc@cse.ust.hk\n2Department of Chemical and Biological Engineering, The Hong Kong University of\nScience and Technology\n3School of Information Science and Engineering, Xiamen University\nAbstract. Histop'

# function to turn each pdf to smaller chunks using chunk_size and chunk_overlap

In [9]:
#function to splitting get into chunks
#chunk_size: the maximum number of characters in each chunk.
#The number of characters shared between consecutive chunks. This overlap can help maintain context when processing the chunks sequentially. Each chunk “remembers” part of the previous one

def chunk_maker(documents, chunk_size=1000, chunk_overlap=200):
    "Split documents into chunks and return a list of chunks"
    #" " words, "" characters, "\n" newlines, "\n\n" paragraphs
    #it is called recursively because it will keep splitting the chunks until they are smaller than the chunk size
    #it tries to split by paragraphs first, then by newlines, then by spaces, and finally by characters if necessary.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len, separators=["\n\n", "\n", " ", ""]) # recursively chunk documents into smaller pieces based on the specified chunk size and overlap.
    split_documents = text_splitter.split_documents(documents) # Split the documents into chunks using the text splitter.
    print(f"Created {len(split_documents)} chunks from {len(documents)} documents.")
    
    if split_documents:
        print(f"\n Example of a chunk")
        print(f"content:{split_documents[0].page_content[:500]}") # Print the first 500 characters of the first chunk
        print(f"metadata: {split_documents[0].metadata}") # Print the metadata of the first chunk
    return split_documents

chunks = chunk_maker(all_pdf_documents, chunk_size=1000, chunk_overlap=200)

Created 548 chunks from 131 documents.

 Example of a chunk
content:HistGen: Histopathology Report Generation via
Local-Global Feature Encoding and Cross-modal
Context Interaction
Zhengrui Guo1, Jiabo Ma1, Yingxue Xu1, Yihui Wang1, Liansheng Wang3, and
Hao Chen1,2( )
1Department of Computer Science and Engineering, The Hong Kong University of
Science and Technology
jhc@cse.ust.hk
2Department of Chemical and Biological Engineering, The Hong Kong University of
Science and Technology
3School of Information Science and Engineering, Xiamen University
Abstract. Histop
metadata: {'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-07-06T03:43:36+00:00', 'author': '', 'keywords': '', 'moddate': '2026-03-25T22:04:37-04:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/PDFs/0796_paper.pdf', 'total_

# Turn chunks to embedding (using Sentence Transformer) and store them in vector Database for retrieval

In [10]:
#using open source model to create text embedding out of chunks and store them in vector database for retrieval

class embed_manager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """ initialize the embeddding model
        Args : model_name = sentencetransformer  model name for sentence embedding generation
        """
        self.model_name = model_name
        self.model = None
        self.load_model()
        
    def load_model(self):
        """Load the embedding model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model :{self.model_name} loaded successfully.")
            print(f" Model dimenions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            
    def embed_generator(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.
        Args:
            texts (List[str]): A list of strings to be embedded.
        Returns:
            np.ndarray: A 2D array where each row is the embedding of the corresponding text.
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        
        print(f"generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print("Embeddings generated successfully.")
        return embeddings
            
##initialize embedding manager

embed_manager = embed_manager()
embed_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/data/users4/pafshin1/TMPDIR/envs/dinov3_env/lib/python3.10/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Model :all-MiniLM-L6-v2 loaded successfully.
 Model dimenions: 384


# Store Embeddings in Vector Store which generates and embedding for each vector and compare distance or similarity using L2 distance or cosine


In [14]:
#“A manager that handles storing and retrieving embeddings (vectors).”
# collection_name: str = "pdf_documents" - the name of the collection in the vector database where the embeddings will be stored.
# persist_directory: str = "./vector_store" - the directory where the vector database will be persisted (saved to disk). This allows the database to be reloaded later without losing data.
import os
import uuid
from typing import List, Any
import numpy as np
import chromadb


class Vector_Store:
    """
    A manager for storing and retrieving embeddings using ChromaDB.
    """

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "./vector_store",
    ):
        """
        Initialize the vector store.

        Args:
            collection_name (str): Name of the collection in ChromaDB.
            persist_directory (str): Path where the DB is stored on disk.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = None
        self.collection = None

        self._initialize_vector_store()

    def _initialize_vector_store(self):
        """
        Create directory, connect to ChromaDB, and initialize collection.
        """

        # 1. Ensure persistence directory exists
        os.makedirs(self.persist_directory, exist_ok=True)
        print(f"Created directory: {self.persist_directory}")

        # 2. Create persistent Chroma client (data saved to disk)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        print("Connected to ChromaDB.")

        # 3. Create or load collection
        # self.collection = self.client.get_or_create_collection(
        #     name=self.collection_name,
        #     metadata={"description": "PDF embeddings for RAG"},
        # )
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "PDF embeddings for RAG",
                "hnsw:space": "cosine"
            },
        )
        print(f"Initialized collection: {self.collection_name}")

        # 4. Show current number of stored documents
        print(f"Existing docs: {self.collection.count()}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents + embeddings into the vector store.

        Args:
            documents (List[Any]): List of document objects (with metadata + page_content)
            embeddings (np.ndarray): Embedding vectors (same length as documents)
        """

        if len(documents) != len(embeddings):
            raise ValueError("Documents and embeddings must have the same length.")

        print(f"Adding {len(documents)} documents...")

        ids = []
        texts = []
        metadatas = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Create a unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Extract text content
            text = doc.page_content
            texts.append(text)

            # Copy and enrich metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(text)
            metadatas.append(metadata)

            # Store embedding
            embeddings_list.append(embedding.tolist() if isinstance(embedding, np.ndarray) else embedding)

        # Add everything to Chroma
        self.collection.add(
            ids=ids,
            documents=texts,
            metadatas=metadatas,
            embeddings=embeddings_list,
        )

        print(f"Total docs after insert: {self.collection.count()}")

    def query(self, query_embedding: List[float], n_results: int = 3):
        """
        Retrieve similar documents using cosine similarity.

        Args:
            query_embedding (List[float]): Embedding of the query.
            n_results (int): Number of results to return.
        """

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
        )

        return results




vector_store = Vector_Store()

vector_store       
print("Number of documents:", vector_store.collection.count())            
        
        

Created directory: ./vector_store
Connected to ChromaDB.
Initialized collection: pdf_documents
Existing docs: 0
Number of documents: 0


In [16]:
import chromadb

client = chromadb.PersistentClient(path="/data/users4/pafshin1/LLM_VLM_NLP/RAG/notebook/vector_store")

collection = client.get_collection(name="pdf_documents")

print(collection.count())

0


In [17]:
chunks[0].page_content

'HistGen: Histopathology Report Generation via\nLocal-Global Feature Encoding and Cross-modal\nContext Interaction\nZhengrui Guo1, Jiabo Ma1, Yingxue Xu1, Yihui Wang1, Liansheng Wang3, and\nHao Chen1,2(\x00)\n1Department of Computer Science and Engineering, The Hong Kong University of\nScience and Technology\njhc@cse.ust.hk\n2Department of Chemical and Biological Engineering, The Hong Kong University of\nScience and Technology\n3School of Information Science and Engineering, Xiamen University\nAbstract. Histopathology serves as the gold standard in cancer diag-\nnosis, with clinical reports being vital in interpreting and understanding\nthis process, guiding cancer treatment and patient care. The automa-\ntion of histopathology report generation with deep learning stands to\nsignificantly enhance clinical efficiency and lessen the labor-intensive,\ntime-consuming burden on pathologists in report writing. In pursuit of\nthis advancement, we introduceHistGen, a multiple instance learning

In [46]:
#convert the text to embedding
texts = [doc.page_content for doc in chunks] 



In [19]:
##generate embeddings for the texts

embeddings = embed_manager.embed_generator(texts)
embeddings.shape

generating embeddings for 548 texts...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Embeddings generated successfully.


(548, 384)

In [20]:
vector_store.add_documents(chunks, embeddings)

Adding 548 documents...
Total docs after insert: 548


In [21]:
import chromadb

client = chromadb.PersistentClient(path="/data/users4/pafshin1/LLM_VLM_NLP/RAG/notebook/vector_store")

collection = client.get_collection(name="pdf_documents")

print(collection.count())

548


# RAG Retrival 


In [22]:
class rag_retriever:
    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5):
        print(f"\n🔍 Query: {query}")

        # Generate embedding
        query_embedding = self.embedding_manager.embed_generator([query])[0]

        try:
            results = self.vector_store.query(
                query_embedding=query_embedding,
                n_results=top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc, meta, dist, doc_id) in enumerate(zip(documents, metadatas, distances, ids)):
                    similarity = 1- dist
                    retrieved_docs.append({
                        'id': doc_id,
                        'content': doc,
                        'metadata': meta,
                        'distance': dist,   
                        'similarity': similarity,
                        'rank': i + 1
                    })

                print(f"✅ Retrieved {len(retrieved_docs)} documents")

                # 🔍 DEBUG: print retrieved docs
                for i, doc in enumerate(retrieved_docs):
                    print(f"\n--- Doc {i+1} (rank {doc['rank']}) ---")
                    print(doc['content'][:300])

            else:
                print("❌ No documents retrieved")

            return retrieved_docs

        except Exception as e:
            print(f"❌ Retrieval error: {e}")
            return []
        
rag_retriever = rag_retriever(vector_store, embed_manager)
rag_retriever               
            
            
        

In [23]:
rag_retriever.retrieve("is the paper targeted the pathology report generation?")



🔍 Query: is the paper targeted the pathology report generation?
generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully.
✅ Retrieved 5 documents

--- Doc 1 (rank 1) ---
2 P. Chen et al.
In spite of the ”clinical-grade” performance of these computational pathol-
ogy approaches, pathologists still need to organize the findings and write textual
reports for each slide. Hundreds to thousands of WSIs need to be summarized in
text in the pathology departments every day [

--- Doc 2 (rank 2) ---
on the epoch with the lowest validation loss.
3 Results
3.1 Report Generation Performance
To evaluate the quality of the generated reports, we performed a reader study. A
pathologist (G.B.) with experience in dermatopathology was recruited to inde-

--- Doc 3 (rank 3) ---
2 Z. Guo et al.
last decade, the increasing availability of digital WSIs has given rise to computa-
tional pathology (CPath), promoting diagnostics with computer-assisted tools.
The recent success of deep learning has notably advanced this field, outper-
forming experienced pathologists on specific 

--- Doc 4 (rank 4

[{'id': 'doc_bf533624_322',
  'content': '2 P. Chen et al.\nIn spite of the ”clinical-grade” performance of these computational pathol-\nogy approaches, pathologists still need to organize the findings and write textual\nreports for each slide. Hundreds to thousands of WSIs need to be summarized in\ntext in the pathology departments every day [9]. The automation of diagnostic\nreports can largely reduce the workload of pathologists. Furthermore, the content\nof pathology reports usually includes abundant diagnostic results [3]. Therefore,\nit motivates us to take a step forward to achieve the automatic generation of\npathology reports. On the data end, the great advancement of computational\npathology in the past years owes very much to the emergence of publicly avail-\nable pathology datasets. Some researchers resort to books, articles, and webs\n[10,19,12] to obtain large-scale image-text pairs. However, their collected images\nare small patches and the corresponding texts are also l

In [25]:
rag_retriever.retrieve("What is CMC module?")



🔍 Query: What is CMC module?
generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully.
✅ Retrieved 5 documents

--- Doc 1 (rank 1) ---
puts is crucial. Meanwhile, leveraging the model’s accumulated knowledge and
memory from previous iterations is expected to significantly enhance this task.
The CMC module (Fig. 1b) is proposed in light of the above insights, denoted
as C = {C1, C2, · · · , Cm}, where Ci is the context vector stored

--- Doc 2 (rank 2) ---
age learned knowledge from the training data that might not be captured by the
input embeddings alone. Similar interactions are applied to textual embeddings
{y1, y2, · · · , yt} from the decoder, except for prototype learning. Additionally,
the CMC module serves as an external memory, enabling the 

--- Doc 3 (rank 3) ---
Table 13: GNN vs. alternative interaction models (PLIP, 16-shot).
Module TCGA NSCLC TCGA BRCA
Intra-scale Hierarchical ACC AUC Macro F1 ACC AUC Macro F1
MLP MLP 69.04±10.5576.15±8.5665.69±16.73 63.91±2.5058.88±3.5158.96±3.52
Attention Attention 71.62±9.8378.54±7.9668

[{'id': 'doc_8137d111_17',
  'content': 'puts is crucial. Meanwhile, leveraging the model’s accumulated knowledge and\nmemory from previous iterations is expected to significantly enhance this task.\nThe CMC module (Fig. 1b) is proposed in light of the above insights, denoted\nas C = {C1, C2, · · · , Cm}, where Ci is the context vector stored in this module.\nFor visual inputs{X ′\n1, · · · , X′\nN }, we select key patches{p1, · · · , pl} via uniform\nsampling-based prototype learning, which aims to use representative patches as\nqueries for the CMC module, addressing redundancy and computational com-\nplexity of the patch sequence. After querying, the responses generated by the\nCMC module are aggregated back into the original visual features, infusing\nthem with cross-modal information. This step aims to help the model lever-\nage learned knowledge from the training data that might not be captured by the\ninput embeddings alone. Similar interactions are applied to textual embeddings'

In [31]:
from dotenv import load_dotenv
import os

load_dotenv("/data/users4/pafshin1/LLM_VLM_NLP/RAG/env/api.env")

print("API KEY FOUND:", os.getenv("OPENAI_API_KEY") is not None)

API KEY FOUND: True


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0
)

In [33]:
response = llm.invoke("Say hello")
print(response.content)

Hello! How can I help you today?


# Rag Enhanced

In [35]:
def enhanced_rag(query, retriever, llm, top_k=3, min_score=0.2):
    
    # Retrieve documents
    results = retriever.retrieve(
        query, 
        top_k=top_k
    )
    
    if not results:
        return {
            'answer': 'No relevant documents found.',
            'sources': [],
            'confidence': 0.0,
            'context': []
        }
    
    # ✅ Deduplicate (optional but good)
    unique_results = []
    seen = set()

    for doc in results:
        content = doc['content'].strip()
        if content not in seen:
            seen.add(content)
            unique_results.append(doc)

    results = unique_results

    # ✅ Context as LIST (for RAGAS)
    context = [doc['content'] for doc in results]

    # Build sources
    sources = [{
        'source': doc.get('metadata', {}).get('source_file', 'unknown'),
        'page': doc.get('metadata', {}).get('page', 'unknown'),
        'score': doc.get('similarity', 0),
        'preview': doc.get('content', '')[:300] + '...'
    } for doc in results]

    # Confidence
    confidence = max(doc.get('similarity', 0) for doc in results)

    # ✅ Join ONLY for LLM
    prompt_context = "\n\n".join(context)

    prompt = f"""
Use only the following context to answer the question concisely in two sentences without using formula.

Context:
{prompt_context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence,
        'context': context   # ✅ KEEP AS LIST
    }

In [37]:
result = enhanced_rag(
    "Why they used region token in the paper? what does it learn?",
    rag_retriever,
    llm,
    top_k=5,
    min_score=0.2
)

print("Answer:", result['answer'])

print("Sources:", result['sources'])

print("Confidence:", result['confidence'])

print("Context:", result['context'][:500], "...")


🔍 Query: Why they used region token in the paper? what does it learn?
generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully.
✅ Retrieved 5 documents

--- Doc 1 (rank 1) ---
Inthiswork,weproposethelocal-globalhierarchicalencoder(Fig.1a)forregion-
aware representation learning. Specifically, we first segment the WSI intoN
regions Xi = {xi,1, xi,2, xi,S}, i = 1 , · · · , N with the region size S. Besides,
we additionally add a region tokenxi,S+1 at the end of each region 

--- Doc 2 (rank 2) ---
for knowledge retrieval. Specifically, the top-k patches with the highest attention
scores are selected, wherek is the selecting ratio, and their corresponding image
embeddings in XPLIP are collected asP = {pi}M ×k
i=1 , where pi ∈ Rd.
Although top-k patches are selected, similarity measure between 

--- Doc 3 (rank 3) ---
k).(3)
The complete set of region-level textual features{t i
k}K
k=1 is
defined ast sub ∈R K×d. We preserve the full-text caption
and encode it into a global textual representationti ∈R d as:
ti =f t([CLS], T i).(4)
During training, subcaptions are randomly sampled f

In [38]:
import pandas as pd
info_query_answer_dir = "/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/Questions_GT/ragas_10_new_questions.csv"
df =pd.read_csv(info_query_answer_dir)
df

,Question,Context,Answer,Ground-Truth
0,What is the CMC module?,NaN,NaN,The cross-modal context module facilitates int...
1,What is the role of attentive pooling?,NaN,NaN,Attentive pooling aggregates the processed fea...
2,What is the Region Q-Former?,NaN,NaN,The Region Q-Former extracts fine-grained embe...
3,What is the role of the visual token cross-att...,NaN,NaN,The visual token cross-attention module iterat...
4,What is the knowledge retrieval mechanism?,NaN,NaN,The knowledge retrieval mechanism fetches sema...
5,What is the bi-modal concurrent learning strat...,NaN,NaN,The bi-modal concurrent learning strategy uses...
6,What is the hierarchical position-aware module?,NaN,NaN,The hierarchical position-aware module strengt...
7,What is the Modality-Scale Attention mechanism?,NaN,NaN,The Modality-Scale Attention mechanism process...
8,What is the Text-Guided Dynamic Filtering module?,NaN,NaN,The Text-Guided Dynamic Filtering module remov...
9,What is text-conditioned region feature alignm...,NaN,NaN,Text-conditioned region feature alignment esta...


In [39]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import json

df = pd.read_csv("/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/Questions_GT/ragas_10_new_questions.csv")

questions = df['Question'].tolist()
ground_truths = df['Ground-Truth'].tolist()

# Pre-allocate results
results_list = [None] * len(questions)


def process_question(i, q):
    result = enhanced_rag(
        q,
        rag_retriever,
        llm,
        top_k=3,
        min_score=0.2
    )
    return i, q, result


# ✅ Parallel execution with index
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(process_question, i, q) for i, q in enumerate(questions)]

    for future in as_completed(futures):
        i, q, result = future.result()
        results_list[i] = {
            "question": q,
            "answer": result['answer'],
            "context": result['context']
        }


# Build DataFrame in correct order
output_df = pd.DataFrame(results_list)

# Add ground truth (aligned correctly now)
output_df["ground_truth"] = ground_truths

# Save contexts safely
output_df["context"] = output_df["context"].apply(json.dumps)

output_df.to_csv("ragas_ready_dataset.csv", index=False)


🔍 Query: What is the CMC module?
generating embeddings for 1 texts...

🔍 Query: What is the role of attentive pooling?
generating embeddings for 1 texts...

🔍 Query: What is the Region Q-Former?
generating embeddings for 1 texts...

🔍 Query: What is the role of the visual token cross-attention module?
generating embeddings for 1 texts...

🔍 Query: What is the knowledge retrieval mechanism?
generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Embeddings generated successfully.
✅ Retrieved 3 documents

--- Doc 1 (rank 1) ---
J.3 Hierarchical Aggregator Baselines
Scale-aware Attention (SAA).To disentangle the contribution of the modality, we define scale-
aware attention by removing the modality-specific transformation (i.e., using shared weights Wq,
Wk, Wv for all relations). The vectors are computed as qv =W q(hv +s v)

--- Doc 2 (rank 2) ---
outputs over all such relations: hintra
v =MEAN({SAGE (r)(v)|r∈ R intra}). Initial node features hv
are modality-specific embeddings.
Hierarchical Aggregator.To capture hierarchical interactions across both modalities and scales,
we introduceModality-Scale Attention (MSA), an attention mechanism app

--- Doc 3 (rank 3) ---
Figure 6: Comparison of hierarchical aggregator meth-
ods (16-shot).
Effects of Hierarchical Aggregator.As
shown in Figure 6, the Modality-Scale At-
tention (MSA) module consistently outper-
forms all baselines across the evaluation
metrics, including Modality-Aware 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully.
✅ Retrieved 3 documents

--- Doc 1 (rank 1) ---
D Text-Guided Dynamic Filtering Pseudocode
Algorithm 1Text-Guided Dynamic Filtering (TGDF)
Require: Low-scale patch features Z(l) ∈R N×D , low-scale text features E(l) ∈R O×D, high-scale
patch featuresZ (h) ∈R R×D, high-scale text featuresE (h) ∈R S×D, sensitivity parameterα
Ensure:Filtered similari

--- Doc 2 (rank 2) ---
plan to explore adaptive, learnable filtering mechanisms to enhance robustness and transferability.
Additionally, the Hit Ratio metric assumes the correctness of LLM-generated parent–child text
structures. We plan to incorporate expert human evaluation to validate these hierarchies and to assess
the

--- Doc 3 (rank 3) ---
cally consistent hierarchical alignment. Following CoOp [59], each text is prepended withL learnable
tokensv (∗) ∈R L×D, resulting in prompt embeddings:
t(l)
o = [v(l)
1 ]. . .[v (l)
L ][Low-scale Texto]t (h)
o,k = [v(h)
1 ]. . .[v (h)
L ][High-scale Texto,k] (1)
The

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully.
✅ Retrieved 3 documents

--- Doc 1 (rank 1) ---
k).(3)
The complete set of region-level textual features{t i
k}K
k=1 is
defined ast sub ∈R K×d. We preserve the full-text caption
and encode it into a global textual representationti ∈R d as:
ti =f t([CLS], T i).(4)
During training, subcaptions are randomly sampled for
region-level alignment, while 

--- Doc 2 (rank 2) ---
Embed.
Align
Align
Align
Figure 1: Comparison of PathFLIP with previous methods in
vision-language pathology modeling. (a) CLIP-based meth-
ods perform coarse global feature alignment between im-
ages and text. (b) Context-conditioned approaches use tex-
tual cues to guide attention but focus on glo

--- Doc 3 (rank 3) ---
andBis the batch size. We adopt the LogSigmoid loss for
regional contrastive learning:
Lregion = 1
L
LX
l=1
−log( 1
1 +exp(y l · −η <v tc
l ,t sub
l >)),
(6)
wherey l = +1for positive pairs and−1for negative ones,
andηis a learnable temperature parameter. The similar

In [40]:
from datasets import Dataset
import json
import pandas as pd

df = pd.read_csv("/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/Questions_GT/ragas_ready_dataset.csv")

# Convert back to list

df["retrieved_contexts"] = df["context"].apply(json.loads)

dataset = Dataset.from_pandas(df)

In [41]:
df['retrieved_contexts'][0]

['Table 13: GNN vs. alternative interaction models (PLIP, 16-shot).\nModule TCGA NSCLC TCGA BRCA\nIntra-scale Hierarchical ACC AUC Macro F1 ACC AUC Macro F1\nMLP MLP 69.04±10.5576.15±8.5665.69±16.73 63.91±2.5058.88±3.5158.96±3.52\nAttention Attention 71.62±9.8378.54±7.9668.24±15.96 67.76±3.0469.24±4.0364.11±3.82\nGraph MLP 74.21±9.1181.04±7.3671.13±15.18 71.67±3.5780.56±4.5571.03±4.11\nGraph Attention 76.99±3.2384.50±3.4676.89±3.24 73.11±3.8381.41±2.2472.96±3.55\nGraph HGNN 80.13±4.73 87.28±2.76 80.08±4.73 75.21±3.51 83.19±4.72 74.99±3.67\nM.5 Single-Scale vs. Multi-Scale Interaction\nTo assess the contribution of scale-level interaction, we evaluate three configurations: (a) using only\nlow-scale features, (b) using only high-scale features, and (c) combining both through multiscale\nintegration. Table 14 highlights the consistent superiority of the multi-scale configuration over\nboth single-scale variants on the TCGA NSCLC and BRCA datasets. Low-scale features primarily',
 'puts is 

In [44]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from datasets import Dataset
import pandas as pd
import json

# ====================== CONFIGURATION ======================
llm = ChatOpenAI(
    model="gpt-4o-mini",      
    temperature=0.1,
    max_tokens=2048,          # Critical to avoid LLMDidNotFinishException
    timeout=120,
)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ====================== LOAD DATA ======================
df = pd.read_csv("/data/users4/pafshin1/LLM_VLM_NLP/RAG/data/Questions_GT/ragas_ready_dataset.csv")

# Ensure contexts column is list
df["retrieved_contexts"] = df["context"].apply(json.loads)

dataset = Dataset.from_pandas(df)

# ====================== METRICS ======================
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,        # This is the pre-instantiated object
    context_precision,
    context_recall,
    answer_correctness,
)

# Correct way to change strictness (newer Ragas versions)
from ragas.metrics import AnswerRelevancy

answer_relevancy_metric = AnswerRelevancy(
    strictness=1,            
    llm=llm,
    embeddings=embeddings
)

# ====================== RUN EVALUATION ======================
print(f"Starting evaluation on {len(dataset)} samples...")


score = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy_metric,   # Use the configured metric here
        context_precision,
        context_recall,
        answer_correctness,
    ],
    llm=llm,
    embeddings=embeddings,
)

# ====================== SAVE RESULTS ======================
score_df = score.to_pandas()
score_df.to_csv("EvaluationScores.csv", index=False)

print("\n✅ Evaluation completed successfully!")
print(score)
print("\nResults saved to EvaluationScores.csv")

Starting evaluation on 10 samples...


/data/users4/pafshin1/tmp/ipykernel_1632832/3044323287.py:27: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/data/users4/pafshin1/tmp/ipykernel_1632832/3044323287.py:27: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/data/users4/pafshin1/tmp/ipykernel_1632832/3044323287.py:27: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/data/users4/pafshin1/tmp/ipykernel_1632832/3044323287.py:27:

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]


✅ Evaluation completed successfully!
{'faithfulness': 0.8983, 'answer_relevancy': 0.7863, 'context_precision': 0.8250, 'context_recall': 0.7000, 'answer_correctness': 0.5960}

Results saved to EvaluationScores.csv
